# 004 - Level4

## 4. Joins & Unions (Combining Data)

### 🔗 Tipos de joins más comunes (Combinación horizontal)

Los **joins** añaden **columnas** de una tabla a otra basándose en una **relación** entre ellas.

- **INNER JOIN**  
  Obtiene únicamente los **registros que coinciden** en ambas tablas.

- **LEFT JOIN**  
  Mantiene **todos los registros de la tabla izquierda** y solo los coincidentes de la tabla derecha.

- **RIGHT JOIN**  
  Mantiene **todos los registros de la tabla derecha** y solo los coincidentes de la tabla izquierda.

---

### 🧱 Unions (Combinación vertical)

Las **unions** añaden **filas** de una consulta debajo de otra.  
Requieren que ambas consultas tengan el **mismo número y orden de columnas**.

- **UNION**  
  Combina los resultados y **elimina registros duplicados**.

- **UNION ALL**  
  Combina los resultados **manteniendo todos los registros**  
  (es más rápido que `UNION`).

---

### 🔧 Condiciones y reglas

- **Join conditions**  
  Definen **cómo se conectan las tablas**, especificando las columnas clave (llaves) utilizadas en el `JOIN`.

- **Type matching**  
  Tanto en **joins** como en **unions**, los **tipos de datos** de las columnas comparadas o combinadas deben ser **compatibles**  
  (por ejemplo: texto con texto, número con número).



In [42]:
import pandas as pd
import numpy as np
import polars as pl

In [4]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## Ejercicio 4.1: El Join del Campeón (Postgres)

### 🎯 Objetivo

Cruzar la información de los **partidos** con la información general de los **mundiales**, para generar un reporte que muestre **cada partido** junto con el **campeón** de ese mundial.

---

### 🧩 Tu misión en SQL

- Seleccionar las siguientes columnas:
  - `matchid`
  - `stadium`
  - `attendance`
  - `winner`
- Utilizar un **`INNER JOIN`** para unir las tablas.
- Definir la condición de unión por el **año**:
  - `matches.year = cups.year`
- Ordenar los resultados por **`attendance`** de forma **descendente**, para identificar los partidos del campeón con mayor asistencia.

---

### 💡 Tip

Usa **alias** para mantener el código limpio y legible:

```sql
FROM worldcup.matches m
INNER JOIN worldcup.cups c
  ON m.year = c.year


```SQL
SELECT
    m.matchid,
    m.stadium,
    (m.attendance::INTEGER) AS attendance,
    c.winner
FROM worldcup.matches AS m
INNER JOIN worldcup.cups AS c ON m.year = c.year
ORDER BY attendance DESC;
```

In [12]:
df_m = df_matches.copy()
df_c = df_cups.copy()

df_merge = df_m.merge(
    df_c,
    on='year',
    how='inner'
)

df_merge = df_merge.sort_values(by='year', ascending=False)
df_merge['attendance_x'] = pd.to_numeric(df_merge['attendance_x'], errors='coerce')

res_f = df_merge[['matchid','stadium','attendance_x','winner']].copy()
res_f = res_f.rename(columns={'attendance_x': 'attendance'})

res_f = res_f.sort_values(by='attendance', ascending=False)

In [15]:
res_f = pl_matches.join(
    pl_cups.select(['year','winner']),
    on = 'year',
    how = 'inner'
).filter(
    pl.col('attendance').is_not_null()
).select([
    'matchid',
    'stadium',
    pl.col('attendance').cast(pl.Int64, strict=False),
    'winner'
]).sort('attendance', descending=True)

## Ejercicio 4.2: El "Misterio" del LEFT JOIN

### 📐 La lógica del LEFT JOIN

A diferencia del **`INNER JOIN`**, el **`LEFT JOIN`** se asegura de que **todos los registros de la tabla de la izquierda** se mantengan en el resultado final, incluso si **no encuentran coincidencia** en la tabla de la derecha.

En esos casos, los datos faltantes se rellenan con **`NULL`** (o `None` / `NaN`, según la tecnología utilizada).

---

### 🎯 Objetivo del ejercicio

Generar un **reporte de todos los mundiales registrados en la historia** (tabla `cups`), junto con la información de sus **partidos** (tabla `matches`).

---

### 🤔 ¿Por qué usar LEFT JOIN aquí?

Porque si existiera un mundial en la tabla `cups`  
(por ejemplo, **2026**) que aún **no tiene partidos registrados** en la tabla `matches`:

- Un **`INNER JOIN`** lo **eliminaría** del reporte.
- Un **`LEFT JOIN`** lo **mantiene visible**, mostrando los datos del partido como **nulos**.

Este comportamiento es clave en análisis del mundo real, donde los datos suelen estar **incompletos o en construcción**.


```SQL
SELECT
    m.matchid,
    m.stadium
FROM worldcup.cups AS c
LEFT JOIN worldcup.matches m on c.year = m.year;
```

In [19]:
df_m = df_matches.copy()
df_c = df_cups.copy()

df_merge = df_cups.merge(
    df_m,
    on='year',
    how='left'
)

res_f = df_merge[['year', 'winner', 'matchid', 'stadium']]

In [23]:
res_f = pl_cups.join(
    pl_matches.select(['year','matchid','stadium']),
    on = 'year',
    how = 'left'
).select(
    ['year','winner', 'matchid', 'stadium']
)

## Ejercicio 4.3: El Triple Join (La Visión de 360°)

### 📐 La lógica del encadenamiento

Imagina que los **`JOIN`** son **eslabones de una cadena**.  
No puedes conectar el primer eslabón con el tercero directamente; necesitas el segundo para poder unirlos.

En este caso, la relación entre las tablas es la siguiente:

- `cups` se conecta con `matches` mediante la columna `year`.
- `matches` se conecta con `players` mediante la columna `matchid`.

---

### 🎯 Objetivo del ejercicio

Crear un **reporte detallado** que nos permita responder a la siguiente pregunta:

> *"En el mundial que ganó **[Winner]**, se jugó un partido en el estadio **[Stadium]** donde participó el jugador **[Player Name]***."

Este ejercicio busca integrar información de **tres tablas distintas** para obtener una **visión completa (360°)** de los datos.


```SQL
SELECT
    c.year,
    c.winner,
    m.stadium,
    p."Player Name"
FROM worldcup.cups AS c
LEFT JOIN worldcup.matches m on c.year = m.year
LEFT JOIN worldcup.players p on m.matchid::TEXT = p.matchid::TEXT;
```

In [33]:
# 1. Copias de seguridad
df_c = df_cups.copy()
df_m = df_matches.copy()
df_p = df_players.copy()

# 2. Limpieza de IDs (Forzamos a entero antes de pasar a string)
# El 'errors="coerce"' es por si hay algún dato basura, y 'fillna(0)' evita que el .0 regrese
df_m['matchid'] = pd.to_numeric(df_m['matchid'], errors='coerce').fillna(0).astype(int).astype(str)
df_p['matchid'] = pd.to_numeric(df_p['matchid'], errors='coerce').fillna(0).astype(int).astype(str)

# 3. Renombrado Explicito (Para saber de dónde viene cada dato)
df_c = df_c.rename(columns={'winner': 'cup_winner'})
df_m = df_m.rename(columns={'stadium': 'match_stadium'})
df_p = df_p.rename(columns={'player name': 'p_name'})

# 4. Triple Merge Encadenado
# Unimos Cups con Matches, y el resultado con Players
df_final = df_c.merge(
    df_m, on='year', how='left'
).merge(
    df_p, on='matchid', how='left'
)

# 5. Selección final con nombres claros
res_f = df_final[['year', 'cup_winner', 'match_stadium', 'p_name']]


In [37]:
# 1. Preparar Cups con su prefijo
# Seleccionamos year (para unir) y winner (con nombre nuevo)
cups_ready = pl_cups.select([
    pl.col("year"),
    pl.col("winner").alias("cup_winner")
])

# 2. Preparar Matches con su prefijo
matches_ready = pl_matches.select([
    pl.col("year"),
    pl.col("matchid"),
    pl.col("stadium").alias("match_stadium")
])

# 3. Preparar Players con su prefijo
players_ready = pl_players.select([
    pl.col("matchid"),
    pl.col("player name").alias("p_name")
])

# 4. Triple Join Encadenado
res_f = cups_ready.join(
    matches_ready, on="year", how="left"
).join(
    players_ready, on="matchid", how="left"
).select([
    "year", "cup_winner", "match_stadium", "p_name"
])

## Ejercicio 4.4: El Gran Consolidado (UNION)

### 🧱 La lógica de la pila (Stacking)

A diferencia del **Triple Join**, donde se añaden **columnas hacia la derecha**, en un **`UNION`** se **apilan filas hacia abajo**.

Imagina que tienes dos reportes separados de **goleadores históricos**:

- **Tabla A**: Goleadores de los mundiales de la **era antigua** (1930 - 1950).
- **Tabla B**: Goleadores de los mundiales de la **era moderna** (2010 - 2014).

---

### 🎯 Objetivo del ejercicio

Construir una **fuente única de goleadores** que combine ambas eras en **una sola lista**, permitiendo analizar quién es el **máximo anotador histórico**, sin importar el año.


```SQL
SELECT "Player Name", "Team Initials", event FROM worldcup.players WHERE event LIKE '%G%' -- G de Goal
-- Combinamos verticalmente
UNION ALL
SELECT "Player Name", "Team Initials", event FROM worldcup.players WHERE event LIKE '%PK%' -- PK de Penalty
```

In [40]:
df_p = df_players.copy()

goles_jugada = df_p[df_p['event'].str.contains('G', na=False)][['player name', 'team initials', 'event']]

goles_penal = df_p[df_p['event'].str.contains('P', na=False)][['player name', 'team initials', 'event']]

df_union = pd.concat([goles_jugada, goles_penal], axis=0)

df_union_unique = df_union.drop_duplicates()

In [44]:
res_vertical = pl.concat([
    pl_players.filter(pl.col('event').str.contains('G')).select(['player name', 'team initials', 'event']),
    pl_players.filter(pl.col('event').str.contains('P')).select(['player name', 'team initials', 'event'])
])

res_vertical

player name,team initials,event
str,str,str
"""Marcel LANGILLER""","""FRA""","""G40'"""
"""Juan CARRENO""","""MEX""","""G70'"""
"""Andre MASCHINOT""","""FRA""","""G43' G87'"""
"""Lucien LAURENT""","""FRA""","""G19'"""
"""Tom FLORIE""","""USA""","""G45'"""
…,…,…
"""V. PERSIE""","""NED""","""P3'"""
"""HUNTELAAR""","""NED""","""I76' P90'"""
"""HUNTELAAR""","""NED""","""I76' P90'"""
